# The Beam Propagation Method
In this notebook you will work through how to augment the angular spectrum method into the more versatile beam propagation method. To practice this, we will use a graded index fiber as a sample medium to propagate through. 

A graded index fiber has a refractive index given by the formula
$$
n(x,y) = n_0 \left(1 - C (x^2 + y^2)\right)
$$
for curvature constant C which encodes the rate of index change in the fiber. This index change leads to a focusing effect where light that has traveled away from the core of the fiber experiences a lower refractive index and as a consequence accelerates, leading to a rotation of the wavefront back towards the core. 

You will walk through the process of implementing BPM by (1) applying a phase mask for accumulated phase due to refractive index, (2) applying a boundary condition in between each propagation step to prevent aliasing effects, and (3) breaking angular spectrum propagation down into multiple, small, steps with the boundary condition and phase mask applied in between each step.



In [18]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider

N = 256
dx = 1e-6
wavelength = 633e-9
L = 10e-3   # total propagation distance

x = (np.arange(N) - N//2) * dx
X, Y = np.meshgrid(x, x)

fx = np.fft.fftfreq(N, dx)
FX, FY = np.meshgrid(fx, fx)

def angular_spectrum_step(field, dz):
    """
    Propagate a field one step using ASM.
    """
    k = 2 * np.pi / wavelength

    kx = 2 * np.pi * FX
    ky = 2 * np.pi * FY

    kz = np.sqrt(k**2 - kx**2 - ky**2 + 0j)

    H = np.exp(1j * kz * dz)

    F = np.fft.fft2(field)
    F_prop = F * H
    field_next = np.fft.ifft2(F_prop)

    return field_next

def gaussian_beam(waist):
    """
    Generate a Gaussian beam.
    """
    r2 = X**2 + Y**2
    return np.exp(-r2 / waist**2)

# Part 1
Let's begin by generating the phase that will be induced by propagating through the graded index fiber. 

To start, we should generate the refractive index profile of the fiber according to the formula that has been provided. After we know the refractive index of the fiber, we can get the phase delay that propagating a distance dz induces from the relative change in refractive index.

Implement the two below functions to generate these values and visualize the results with the provided plotting code. No discussion is needed for this section, but make sure the results seem reasonable.

In [5]:
def grin_refractive_index(n0, curvature):
    """
    Create a GRIN refractive index profile.

    Parameters
    ----------
    n0 : center refractive index
    curvature : strength of parabolic index

    Returns
    -------
    n : refractive index distribution
    """
    r2 = X**2 + Y**2
    n = n0 * (1 - curvature * r2)

    return n

In [6]:
def phase_delay_from_index(n, dz, n_reference):
    """
    Convert refractive index distribution into phase delay.

    Parameters
    ----------
    n : refractive index map
    dz : propagation step
    n_reference : reference refractive index

    Returns
    -------
    phase_mask : complex phase mask
    """
    k0 = 2 * np.pi / wavelength
    delta_n = n - n_reference
    phase = k0 * delta_n * dz
    phase_mask = np.exp(1j * phase)

    return phase_mask

In [9]:
# - No modification necessary -

def visualize_grin_profile():
    def update(curvature=1e8):
        n0 = 1.45
        dz = 1e-5

        n = grin_refractive_index(n0, curvature)

        phase = phase_delay_from_index(n, dz, n0)

        fig, ax = plt.subplots(1,2, figsize=(10,4))

        im0 = ax[0].imshow(n)
        ax[0].set_title("Refractive Index")
        fig.colorbar(im0, ax=ax[0])

        im1 = ax[1].imshow(np.angle(phase))
        ax[1].set_title("Phase Delay")
        fig.colorbar(im1, ax=ax[1])

        plt.show()

    interact(
        update,
        curvature=FloatSlider(min=5e5,max=25e5,step=5e5,value=1e6)
    )
    
visualize_grin_profile()

interactive(children=(FloatSlider(value=1000000.0, description='curvature', max=2500000.0, min=500000.0, step=…

# Part 2
Now that we have the phase delay that our fiber induces, we will move on to an absorbing boundary condition. An absorbing boundary condition is designed to supress light that is approaching the edge of the simulation domain to prevent it from reflecting and creating aliasing. A common choice for this is to use a super gaussian function, which follows the form
$$
M(x,y) =
\exp\left[-\left(\frac{|x|}{R}\right)^m\right]
\exp\left[-\left(\frac{|y|}{R}\right)^m\right]
$$
The exponent $m$ allows us to control how sharp the transition region between the on and off regions of the mask is, which we can use to make sure we do not supress too much of our actual signal.

Implement the function below and use the provided plotting code to visualize the mask you have created, then answer the following questions:
1) Why would we use something with soft edges like a supergaussian instead of just cutting off anything close to the edge and setting it directly to 0?
2) Empirically, what seems like a good exponent to use for the supergaussian?
3) Why does employing an absorbing mask help with the distance we are able to propagate in our simulation? 

In [10]:
def supergaussian_mask(radius, exponent):
    """
    Generate a supergaussian absorbing mask.

    Parameters
    ----------
    radius : mask radius
    exponent : supergaussian exponent

    Returns
    -------
    mask : absorbing mask
    """
    x = X[0, :]
    y = Y[:, 0]

    mask_x = np.exp(-(np.abs(x) / radius)**exponent)
    mask_y = np.exp(-(np.abs(y) / radius)**exponent)

    mask = np.outer(mask_y, mask_x)

    return mask

In [11]:
# - No modification necessary -

def visualize_absorbing_mask():
    def update(exponent=8):

        mask = supergaussian_mask(radius=112*dx, exponent=exponent)

        plt.figure(figsize=(5,5))
        plt.imshow(mask, cmap="viridis")
        plt.title("Supergaussian Mask")
        plt.colorbar()
        plt.show()

    interact(
        update,
        exponent=IntSlider(min=2,max=8,step=1,value=2)
    )
    
visualize_absorbing_mask()

interactive(children=(IntSlider(value=4, description='exponent', max=8, min=2), Output()), _dom_classes=('widg…

## Discussion
1) The harsher the edges are, the more intense the diffraction they generate will be, as locally they contain much higher spatial frequency components. This means to avoid the effect that our mask, which is in effect an aperture, has on our system, we want to keep the edges soft and low frequency.
2) Something around 5 or so (4-6) seems to strike a good balance of keeping the main content of our light while not having incredibly harsh edges.
3) By using a mask we can prevent reflections that otherwise would have polluted our ASM before they happen. This means that by applying a few ASM steps with that are just shy of the distance required to start producing those reflections with the absorbing mask in between, we should be able to propagate further and not see the same negative effect.

# Part 3
Now that we have everything that we need, let's put together the full BPM routine.

Remember that BPM is in essence just a repeated application of the angular spectrum method. A phase mask is included in between each step to account for the phase accumulated from the relative properties of the medium through which our light is propagating, and an absorbing boundary is applied to remove any escaping light that could become detrimental to the system.

Below you should implement the full BPM routine. The function will return an x-z cutplane, so you should record a cutline at each step of the propagation and combine them to form the returned image.

After you have implemented this, you can move on to Part 4 where there will be plotting code that allows you to visualize the effect of the phase mask and absorbing boundary on propagation.

In [12]:
def bpm_propagate_record(
    field,
    dz,
    total_distance,
    phase_mask=None,
    absorbing_mask=None
):
    """
    Beam propagation method that records the field evolution.

    Returns a side-view intensity map I(x,z).
    """

    steps = int(total_distance / dz)

    center = N // 2

    current = field.copy()

    intensity_xz = np.zeros((steps, N))

    for i in range(steps):

        current = angular_spectrum_step(current, dz)

        if phase_mask is not None:
            current *= phase_mask

        if absorbing_mask is not None:
            current *= absorbing_mask

        intensity_xz[i] = np.abs(current[center])**2

    return intensity_xz

# Part 4
Provided below are plotting routines for 4 configurations:
1) Free space propagation (no phase mask or absorbing boundary
2) Absorbing propagation (no phase mask with absorbing boundary)
3) GRIN propagation (phase mask but no absorbing boundary)
4) GRIN BPM (phase mask and absorbing boundary)

Take some time to play around with all of them, then comment on the following:
1) What do you observe about the required step size to properly sample the simulation and how it changes with fiber refractive index curvature? Why do you think the curvature effects what values of dz work?
2) Does the absorbing boundary seem to be important in this simulation? Why do you predict that it is or isn't?

In [13]:
# - No modification necessary -

def plot_side_view(intensity_xz, dz):
    """
    Plot side-view propagation: x vs z
    """

    steps, _ = intensity_xz.shape
    z = np.arange(steps) * dz

    extent = [
        x.min()*1e6,
        x.max()*1e6,
        z.max()*1e3,
        z.min()*1e3
    ]

    plt.figure(figsize=(6,5))

    plt.imshow(
        intensity_xz,
        extent=extent,
        aspect="auto",
        cmap="inferno"
    )

    plt.xlabel("x (µm)")
    plt.ylabel("z (mm)")
    plt.title("Beam Propagation")
    plt.colorbar(label="Intensity")

    plt.show()

In [14]:
def demo_free_space():
    beam = gaussian_beam(waist=20e-6)

    def update(dz=2e-5):
        intensity_xz = bpm_propagate_record(
            beam,
            dz,
            L
        )

        plot_side_view(intensity_xz, dz)

    interact(
        update,
        dz=FloatSlider(min=5e-5, max=5e-4, step=10e-5, value=1e-4, readout_format='.6f'),
    )

demo_free_space()

interactive(children=(FloatSlider(value=0.0001, description='dz', max=0.0005, min=5e-05, readout_format='.6f',…

In [15]:
def demo_absorbing_boundaries():
    beam = gaussian_beam(waist=20e-6)
    
    def update(dz=2e-5, exponent=4):
        mask = supergaussian_mask(radius=112*dx, exponent=exponent)

        intensity_xz = bpm_propagate_record(
            beam,
            dz,
            L,
            absorbing_mask=mask
        )

        plot_side_view(intensity_xz, dz)

    interact(
        update,
        dz=FloatSlider(min=5e-5, max=5e-4, step=10e-5, value=1e-4, readout_format='.6f'),
        exponent=IntSlider(min=2,max=8,value=4)
    )
    
demo_absorbing_boundaries()

interactive(children=(FloatSlider(value=0.0001, description='dz', max=0.0005, min=5e-05, readout_format='.6f',…

In [16]:
def demo_grin_fiber():
    beam = gaussian_beam(waist=20e-6)

    def update(dz=2e-5, curvature=1e8):
        n0 = 1.45
        n = grin_refractive_index(n0, curvature)

        phase_mask = phase_delay_from_index(n, dz, n0)

        intensity_xz = bpm_propagate_record(
            beam,
            dz,
            L,
            phase_mask=phase_mask
        )

        plot_side_view(intensity_xz, dz)

    interact(
        update,
        dz=FloatSlider(min=5e-5, max=5e-4, step=10e-5, value=1e-4, readout_format='.6f'),
        curvature=FloatSlider(min=5e5,max=25e5,step=5e5,value=1e6)
    )

demo_grin_fiber()

interactive(children=(FloatSlider(value=0.0001, description='dz', max=0.0005, min=5e-05, readout_format='.6f',…

In [17]:
def demo_full_bpm():
    beam = gaussian_beam(waist=20e-6)

    def update(dz=2e-5, curvature=1e8, exponent=10):

        n0 = 1.45
        n = grin_refractive_index(n0, curvature)
        phase_mask = phase_delay_from_index(n, dz, n0)

        mask = supergaussian_mask(radius=112*dx, exponent=exponent)

        intensity_xz = bpm_propagate_record(
            beam,
            dz,
            L,
            phase_mask=phase_mask,
            absorbing_mask=mask
        )

        plot_side_view(intensity_xz, dz)

    interact(
        update,
        dz=FloatSlider(min=1e-6, max=5e-4, step=10e-5, value=1e-4, readout_format='.6f'),
        curvature=FloatSlider(min=5e5,max=25e5,step=5e5,value=1e6),
        exponent=IntSlider(min=2,max=8,value=4)
    )
    
demo_full_bpm()

interactive(children=(FloatSlider(value=0.0001, description='dz', max=0.0005, min=1e-06, readout_format='.6f',…

## Discussion
1) For the free space and absorbing boundary propagations, the step size does not seem to be too impactful. In the fiber, however, the stronger the curvature is (and therefore the faster the self-focusing effect), the greater the sampling resolution in z needs to be to properly capture the effect. This is because at greater values of curvature, each step takes on greater relative phase changes for a fixed dz, and so our approximation that the light has propagated according to standard ASM within that region begins to fall apart. To improve our simulation, we must decrease dz and thus the relative phase change.
2) Ultimately in this simulation the absorbing boundary does not play a significant role. This is not because it is not useful, but rather because the graded index fiber itself is focusing, which means that a minimal amount of the light is ever approaching the boundary of the simulation domain.